In [ ]:
# !pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb


In [1]:
from langchain_core.documents import Document

In [2]:
sample_doc = Document(
    page_content = "Hello World",
    metadata = {"source": "https://www.google.com"}
)    

In [3]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World')

In [4]:
type(sample_doc)

langchain_core.documents.base.Document

In [5]:
# text data
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("data/python.txt", encoding = "utf-8")

In [6]:
document = loader.load()

In [7]:
document

[Document(metadata={'source': 'data/python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and mo

In [8]:
# pdf data

# from langchain_community.document_loaders.pdf import PyPDFLoader

# pdf_loader = PyPDFLoader("data/research.pdf")

# document = pdf_loader.load()


In [ ]:
# pdf data

from langchain_community.document_loaders.pdf import PyMuPDFLoader

pdf_loader = PyMuPDFLoader("data/research.pdf")

document = pdf_loader.load()




# Ingestion Pipeline

In [13]:
# data => document
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

### Document

In [14]:

def load_all_pdfs():
    folder_path = "data/pdf"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            #complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

In [15]:
all_pdf_documents = load_all_pdfs()

total pdfs: 2
total pages: 34


In [16]:
type(all_pdf_documents[1])

langchain_core.documents.base.Document

### chunks

In [17]:
# Chunks

#!pip install Langchain_text_splitters

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size = 500, chunk_overlap = 50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [19]:
chunks = split_docs(all_pdf_documents)


In [20]:
len(chunks)

383

### Embedding

In [21]:
from sentence_transformers import SentenceTransformer

In [22]:
class EmbeddingManager:
    def __init__(self, model_name ="all-MiniLM-L6-v2"):

        self.model_name = model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimentions=", self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar = True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

In [23]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding dimentions= 384


# Vector Store

In [24]:
import chromadb
import uuid

In [25]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok = True)

        #create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        #create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collecton:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")

        # store => ids, embeddings, documents, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_lenght"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids = ids,
                metadatas = all_metadata,
                documents = documents_content,
                embeddings = embeddings_list
            )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())
        

In [26]:
vector_store = VectorStoreManager()

initialized the vector store with collecton: pdf_documents
docs in collection: 1532


In [27]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embedding)

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

embeddings shape: (383, 384)
total documents added in vector store= 383
docs in collection: 1915


# retrieval Pipeline

In [28]:
from sklearn.metrics.pairwise import cosine_similarity

In [29]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k = 5, score_threshold=0.0):
        #query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        #semantic search
        results = self.vector_store.collection.query(
            query_embeddings =[query_embeddings.tolist()],
            n_results=top_k
        )

        #cosine similarity
        retrieved_docs=[]

        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadta":metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank": i+1
                    })

            print(f"retrieved{len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [30]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [31]:
rag_retriever.retrieve("What is RAG")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved5 documents


[{'id': 'doc_35282981-4ed4-4d5e-8cab-840c70261533',
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'metadta': {'source': 'data/pdf\\research2.pdf',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'doc_index': 150,
   'creator': 'LaTeX with hyperref',
   'subject': '',
   'total_pages': 21,
   'trapped': '/False',
   'content_lenght': 288,
   'page_label': '1',
   'producer': 'pdfTeX-1.40.25',
   'title': '',
   'moddate': '2024-03-28T00:54:45+00:00',
   'author': '',
   'creationdate': '2024-03-28T00:54:45+00:00',
   'page': 0,
   'keywords': ''},
  'distance': 0.4226756691932678,
  'similarity_score': 0.5773243308067322,
  'rank': 1},
 {'id': 'doc_7837c850-8

# Integrate with LLMs

# GROQ

In [64]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [65]:
API_KEY_GROQ = os.getenv("GROQ_API_KEY")

In [ ]:
# !pip install langchain-groq

In [67]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key = API_KEY_GROQ,
    temperature = 0.1,
    model="qwen/qwen3-32b" ,
    max_tokens = 1024
)

In [68]:
 #generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke([prompt.format(context=context, query=query)]) # expecting a list as prompt
    return response.content

In [70]:
answer = generate_output("what is RAG", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved3 documents


In [49]:
print(answer)

<think>
Okay, the user is asking, "What is RAG?" Let me start by recalling what I know about RAG. From the context provided, it's mentioned in a survey that reviews state-of-the-art RAG methods. The context repeats the phrase "speculate on upcoming trends and innovations" and mentions paradigms like naive RAG. The arXiv reference is from March 2024, so it's a recent paper.

First, I need to define RAG. RAG stands for Retrieval-Augmented Generation. It's a method that combines retrieval from a large corpus with a generative model. The basic idea is that instead of relying solely on the model's internal knowledge, it retrieves relevant documents from an external source and uses them to generate more accurate and up-to-date responses.

The context mentions "naive RAG" as a paradigm. I should explain that naive RAG is the simplest form where the model retrieves documents and then generates an answer based on them. However, there are more advanced paradigms beyond the naive approach, which 

# Gemini API

In [71]:
# !pip install -U Langchain-google-genai

In [73]:
 # !pip install "google-genai==0.8.0" "langchain-google-genai==2.0.10"

In [74]:
API_KEY_GEMINI = os.getenv("GEMINI_API_KEY")

In [75]:
from langchain_google_genai import ChatGoogleGenerativeAI

gemini_model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",  # lightest, least quota usage
    google_api_key=API_KEY_GEMINI,
    temperature=0.1,
    max_tokens=1024
)
from langchain_google_genai import ChatGoogleGenerativeAI


In [76]:
from langchain_core.messages import HumanMessage

def generate_output_gemini(query, rag_retriever, llm, top_k=3):
    results = retriever.retreive(query, top_k)
    context = "\n".join(doc["document"] for doc in results) if results else ""
    
    if not context:
        print("No relevant context found for the given query")
    
    prompt = f"""Use the given context to answer the query.
Context: {context}
Query: {query}"""
    
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content

In [77]:
answer = generate_output("RAG Robustness", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved3 documents


In [78]:
print(answer)

<think>
Okay, the user is asking about RAG robustness. Let me start by recalling what RAG is. RAG stands for Retrieval-Augmented Generation, which combines retrieval of documents with a generative model to answer queries. The context provided mentions a survey on RAG methods, their evolution, and upcoming trends. The user wants to know about robustness in RAG systems.

First, I need to define what robustness means in this context. Robustness here likely refers to the system's ability to handle various challenges like noisy data, adversarial attacks, or distribution shifts. The survey probably discusses different paradigms of RAG, such as naive RAG, and how each addresses robustness.

Looking at the context, the survey is thorough and systematic, so it might cover existing methods to enhance robustness. Common issues in RAG include retrieval errors, where the retrieved documents are irrelevant or incomplete. Techniques like using multiple retrieval sources, filtering, or re-ranking coul